In [1]:
import cv2
import pickle

recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("face_recognizer.yml")

with open("labels.pickle", "rb") as f:
    label_map = pickle.load(f)
id_to_name = {v: k for k, v in label_map.items()}

face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
print("Model and labels loaded:", id_to_name)

Model and labels loaded: {0: 'Atul Singh', 1: 'Pratham_Angra', 2: 'Shubhransu_Barik'}


In [2]:
import csv
import os
from datetime import datetime

CONFIDENCE_THRESHOLD = 55

def mark_attendance(name):
    today = datetime.now().strftime("%Y-%m-%d")
    file_path = f"{today}.csv"

    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            for row in csv.reader(f):
                if row and row[0] == name:
                    return False

    file_exists = os.path.exists(file_path)
    with open(file_path, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Name", "Time"])
        writer.writerow([name, datetime.now().strftime("%H:%M:%S")])
    return True

print("mark_attendance ready")

mark_attendance ready


In [3]:
import requests
import threading

LAPTOP_IP = "192.168.2.1"

def send_result(name, box):
    # run the network call in a background thread so recognition doesn't pause and wait
    def _send():
        try:
            requests.post(
                f"http://{LAPTOP_IP}:5000/update_result",
                json={"name": name, "box": list(box)},
                timeout=1
            )
        except Exception:
            pass
    threading.Thread(target=_send, daemon=True).start()

In [4]:
import time

# tracks which face we're currently verifying, and since when
candidate_name = None
candidate_since = None
VERIFY_SECONDS = 1.5  # face must be seen continuously this long before marking

cap = cv2.VideoCapture(f"http://{LAPTOP_IP}:5000/video")
print("Running continuously. Click the ■ (Interrupt) button above to stop.")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to read frame")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(30, 30))

        # only handle the single largest face (avoids double-counting one person)
        if len(faces) > 0:
            faces = [max(faces, key=lambda f: f[2] * f[3])]
        else:
            candidate_name = None
            candidate_since = None
            send_result(None, (0, 0, 0, 0))

        for (x, y, w, h) in faces:
            face_crop = gray[y:y+h, x:x+w]
            label_id, confidence = recognizer.predict(face_crop)

            if confidence < CONFIDENCE_THRESHOLD:
                name = id_to_name.get(label_id, "Unknown")
            else:
                name = "Unknown"

            if name == "Unknown":
                candidate_name = None
                candidate_since = None
                send_result("Unknown", (int(x), int(y), int(w), int(h)))
                continue

            # new/different face -> restart the verification timer
            if name != candidate_name:
                candidate_name = name
                candidate_since = time.time()

            elapsed = time.time() - candidate_since

            if elapsed >= VERIFY_SECONDS:
                # seen continuously long enough -> safe to mark
                marked = mark_attendance(name)
                if marked:
                    print(f"Attendance marked: {name}")
                # always show this once verified, whether marked just now or earlier today
                send_result(f"{name} - Attendance Marked!", (int(x), int(y), int(w), int(h)))
            else:
                # still counting down, show progress on laptop's live window
                remaining = VERIFY_SECONDS - elapsed
                send_result(f"Verifying {name}... {remaining:.1f}s", (int(x), int(y), int(w), int(h)))

except KeyboardInterrupt:
    pass
finally:
    cap.release()
    print("Stopped.")

Running continuously. Click the ■ (Interrupt) button above to stop.
Attendance marked: Atul Singh
Failed to read frame
Stopped.


In [5]:
from datetime import datetime
print(datetime.now())

2026-08-03 12:30:40.277870
